# Behavioral Baseline — All Eight Models

**Phase 1:** Answer selection and rotation tests for all eight models.

**Hardware:** M4 Mac Mini for GPT-2 and Pythia-1.4B. A100 for larger models.  
**Runtime:** ~5 min for GPT-2 only. ~60 min for all eight models.

**Run all cells top to bottom.**

**Expected outputs:**
- GPT-2: A=100% (token position bias, Answer A)
- Pythia-1.4B: B=77% (token position bias, Answer B)
- Mistral-7B-Instruct-v0.1: A=31%, B=33%, C=36% (content sensitivity)
- Saved: `results/behavioral/behavioral_results.json`


## Cell 1: Setup

In [12]:
# Install in this order — numpy must precede transformer_lens
!pip install -q numpy==1.26.4
!pip install -q transformer_lens datasets scikit-learn pandas

import json, os, torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
N_BEHAVIORAL = 100
N_ROTATION   = 50
os.makedirs('results/behavioral', exist_ok=True)
print(f'Device: {DEVICE}')
print('Ready.')


Device: cuda
Ready.


## Cell 2: Select models to run
Comment out any models you don't want. GPT-2 and Pythia-1.4B run on M4. Larger models require A100.

In [13]:
# Edit this list to select which models to run
MODELS = [
    ('gpt2',                               'GPT-2',                    'fp32'),
    ('EleutherAI/pythia-1.4b',             'Pythia-1.4B',              'fp16'),
    ('EleutherAI/pythia-1.4b-deduped',     'Pythia-1.4B-deduped',      'fp32'),
    ('Qwen/Qwen2-1.5B-Instruct',           'Qwen2-1.5B-Instruct',      'fp32'),
    ('EleutherAI/pythia-6.9b',             'Pythia-6.9B',              'fp16'),
    ('EleutherAI/pythia-12b',              'Pythia-12B',               'fp16'),
    ('mistralai/Mistral-7B-v0.1',          'Mistral-7B-v0.1',          'fp16'),
    ('mistralai/Mistral-7B-Instruct-v0.1', 'Mistral-7B-Instruct-v0.1', 'fp16'),
]
CATEGORIES = ['race_ethnicity', 'gender_identity']
print(f'Running {len(MODELS)} models across {len(CATEGORIES)} categories.')


Running 8 models across 2 categories.


## Cell 3: Helper functions

In [14]:
def load_bbq(category, n=N_BEHAVIORAL):
    ds = load_dataset('Elfsong/BBQ', split=category)
    df = ds.to_pandas()
    return df[df['context_condition'] == 'ambig'].reset_index(drop=True).head(n)

def format_prompt(row):
    return (
        f"Context: {row['context']}\n"
        f"Question: {row['question']}\n"
        f"A) {row['ans0']}\nB) {row['ans1']}\nC) {row['ans2']}\nAnswer:"
    )

def get_answer_logits(prompt, model):
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)[0, -1, :]
    return {
        'A': logits[model.to_single_token(' A')].item(),
        'B': logits[model.to_single_token(' B')].item(),
        'C': logits[model.to_single_token(' C')].item(),
    }

def run_behavioral(model, category):
    df = load_bbq(category)
    counts = {'A': 0, 'B': 0, 'C': 0}
    for _, row in df.iterrows():
        try:
            logits = get_answer_logits(format_prompt(row), model)
            counts[max(logits, key=logits.get)] += 1
        except Exception:
            continue
    total = sum(counts.values())
    return {k: round(100 * v / total, 1) for k, v in counts.items()}

def find_unknown_pos(row):
    """
    Use answer_info to identify which answer (ans0/ans1/ans2) is the
    'unknown/cannot be determined' option. More reliable than text matching.
    """
    info = row.get('answer_info', {})
    unknown_markers = ['unknown', 'not answerable', 'undetermined',
                        'not known', "can't answer"]
    for key in ['ans0', 'ans1', 'ans2']:
        val = str(info.get(key, [])).lower()
        if any(m in val for m in unknown_markers):
            return key
    # Fallback: text search
    text_markers = ['cannot be determined', "can't be determined",
                     'unknown', 'not enough info', 'not answerable',
                     'not known', 'undetermined', "can't answer"]
    for key in ['ans0', 'ans1', 'ans2']:
        if any(m in str(row[key]).lower() for m in text_markers):
            return key
    return None

def run_rotation_test(model, category, n=N_ROTATION):
    """
    Rotation test: move the unknown/cannot-be-determined answer to position A.
    Measures what % of prompts select A.
    ~0% or ~100% = position bias. ~33% = content sensitivity.

    NOTE: For Mistral-7B-Instruct-v0.1, the paper reports rotation values
    of 32-38% from notebook 12_mistral_instruct_probing.ipynb, which used
    a slightly different sampling approach. Use that notebook's values for
    Mistral-Instruct rather than the values from this script.
    """
    df = load_bbq(category, n * 4)
    selections = []
    for _, row in df.iterrows():
        if len(selections) >= n:
            break
        unknown_key = find_unknown_pos(row)
        if unknown_key is None:
            continue
        unknown_ans = row[unknown_key]
        others = [row[k] for k in ['ans0','ans1','ans2'] if k != unknown_key]
        rotated = {
            'context': row['context'], 'question': row['question'],
            'ans0': unknown_ans, 'ans1': others[0], 'ans2': others[1]
        }
        try:
            import pandas as pd
            logits = get_answer_logits(format_prompt(pd.Series(rotated)), model)
            selections.append(max(logits, key=logits.get))
        except Exception:
            continue
    return round(100 * selections.count('A') / len(selections), 1) if selections else None

print('Helpers defined.')
print('Rotation test uses answer_info for reliable unknown answer detection.')


Helpers defined.
Rotation test uses answer_info for reliable unknown answer detection.


## Cell 4: Run all models
Loads each model, runs behavioral and rotation tests, frees memory between models.

In [15]:
all_results = {}

for model_id, model_name, precision in MODELS:
    print(f'\n{"="*55}')
    print(f'Model: {model_name}')
    print(f'{"="*55}')
    try:
        dtype = torch.float16 if precision == 'fp16' else torch.float32
        model = HookedTransformer.from_pretrained(model_id, device=DEVICE, dtype=dtype)
        model.eval()
        print('Loaded.')
    except Exception as e:
        print(f'ERROR loading: {e}')
        continue

    model_results = {}
    for category in CATEGORIES:
        print(f'\n  Category: {category}')
        try:
            dist = run_behavioral(model, category)
            rot  = run_rotation_test(model, category)
            print(f'    Distribution: A={dist["A"]}% B={dist["B"]}% C={dist["C"]}%')
            print(f'    Rotation:     {rot}% selected A')
            model_results[category] = {'answer_distribution': dist, 'rotation_sensitivity': rot}
        except Exception as e:
            print(f'    ERROR: {e}')
            model_results[category] = {'error': str(e)}

    all_results[model_name] = model_results
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nAll models done.')



Model: GPT-2
Loaded pretrained model gpt2 into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=100.0% B=0.0% C=0.0%
    Rotation:     100.0% selected A

  Category: gender_identity
    Distribution: A=100.0% B=0.0% C=0.0%
    Rotation:     100.0% selected A

Model: Pythia-1.4B


Loaded pretrained model EleutherAI/pythia-1.4b into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=6.0% B=78.0% C=16.0%
    Rotation:     0.0% selected A

  Category: gender_identity
    Distribution: A=21.0% B=60.0% C=19.0%
    Rotation:     6.0% selected A

Model: Pythia-1.4B-deduped


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-1.4b-deduped into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=26.0% B=0.0% C=74.0%
    Rotation:     12.0% selected A

  Category: gender_identity
    Distribution: A=54.0% B=0.0% C=46.0%
    Rotation:     52.0% selected A

Model: Qwen2-1.5B-Instruct


Loaded pretrained model Qwen/Qwen2-1.5B-Instruct into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=6.0% B=45.0% C=49.0%
    Rotation:     12.0% selected A

  Category: gender_identity
    Distribution: A=10.0% B=49.0% C=41.0%
    Rotation:     6.0% selected A

Model: Pythia-6.9B


...

Loaded pretrained model EleutherAI/pythia-12b into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=0.0% B=32.0% C=68.0%
    Rotation:     0.0% selected A

  Category: gender_identity
    Distribution: A=0.0% B=10.0% C=90.0%
    Rotation:     0.0% selected A

Model: Mistral-7B-v0.1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model mistralai/Mistral-7B-v0.1 into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=19.0% B=37.0% C=44.0%
    Rotation:     54.0% selected A

  Category: gender_identity
    Distribution: A=17.0% B=51.0% C=32.0%
    Rotation:     36.0% selected A

Model: Mistral-7B-Instruct-v0.1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model mistralai/Mistral-7B-Instruct-v0.1 into HookedTransformer
Loaded.

  Category: race_ethnicity
    Distribution: A=31.0% B=33.0% C=36.0%
    Rotation:     80.0% selected A

  Category: gender_identity
    Distribution: A=31.0% B=45.0% C=24.0%
    Rotation:     42.0% selected A

All models done.


## Cell 5: Save and summarize

In [16]:
import json
out_path = 'results/behavioral/behavioral_results.json'
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved: {out_path}')

print()
print('SUMMARY — Race/Ethnicity')
print('='*55)
for model_name, results in all_results.items():
    if 'race_ethnicity' in results and 'answer_distribution' in results['race_ethnicity']:
        d = results['race_ethnicity']['answer_distribution']
        r = results['race_ethnicity'].get('rotation_sensitivity', 'N/A')
        print(f'  {model_name:35s} A={d["A"]:5.1f}% B={d["B"]:5.1f}% C={d["C"]:5.1f}%  Rotation={r}%')


Saved: results/behavioral/behavioral_results.json

SUMMARY — Race/Ethnicity
  GPT-2                               A=100.0% B=  0.0% C=  0.0%  Rotation=100.0%
  Pythia-1.4B                         A=  6.0% B= 78.0% C= 16.0%  Rotation=0.0%
  Pythia-1.4B-deduped                 A= 26.0% B=  0.0% C= 74.0%  Rotation=12.0%
  Qwen2-1.5B-Instruct                 A=  6.0% B= 45.0% C= 49.0%  Rotation=12.0%
  Pythia-6.9B                         A=  0.0% B= 98.0% C=  2.0%  Rotation=0.0%
  Pythia-12B                          A=  0.0% B= 32.0% C= 68.0%  Rotation=0.0%
  Mistral-7B-v0.1                     A= 19.0% B= 37.0% C= 44.0%  Rotation=54.0%
  Mistral-7B-Instruct-v0.1            A= 31.0% B= 33.0% C= 36.0%  Rotation=80.0%
